# 🧠 AirLyst: [03] Feature Engineering
**Mission: Build the Brain**

In this notebook, we transform our cleaned data into powerful features. Time-series models need more than just raw numbers; they need to know what happened in the past (Lags) and general trends (Rolling Averages).

---

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Setup Path
ROOT_DIR = Path("C:/Users/Dell/Desktop/AirLyst")
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

print("Feature Engineering Engine Ready.")

Feature Engineering Engine Ready.


## 📂 1. Load Cleaned Data

In [2]:
data_path = ROOT_DIR / "backend/data/cleaned_historical_data.csv"
df = pd.read_csv(data_path)
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')

print(f"Loaded {len(df)} rows for engineering.")

Loaded 13176 rows for engineering.


## 📅 2. Time-Based Features
These help the model understand seasonal and daily patterns (e.g., morning traffic peaks).

In [3]:
df['hour'] = df['time'].dt.hour
df['day'] = df['time'].dt.day
df['month'] = df['time'].dt.month
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print(" Added Time Features: Hour, Day, Month, DayOfWeek, Weekend.")

 Added Time Features: Hour, Day, Month, DayOfWeek, Weekend.


## 🕒 3. Lag Features
Air quality is highly dependent on what it was a few hours ago. We add 'lags' to represent this memory.

In [4]:
# Lag PM2.5 and US AQI
for target in ['pm2_5', 'us_aqi']:
    df[f'{target}_lag_1h'] = df[target].shift(1)
    df[f'{target}_lag_3h'] = df[target].shift(3)
    df[f'{target}_lag_6h'] = df[target].shift(6)
    df[f'{target}_lag_24h'] = df[target].shift(24)

print(" Added Lag Features (1h, 3h, 6h, 24h) for PM2.5 and AQI.")

 Added Lag Features (1h, 3h, 6h, 24h) for PM2.5 and AQI.


## 🌊 4. Rolling Features
Smoothing the data helps the model see trends rather than just spikes.

In [5]:
# 6-hour and 24-hour moving averages for PM2.5
df['pm2_5_rolling_6h'] = df['pm2_5'].rolling(window=6).mean()
df['pm2_5_rolling_24h'] = df['pm2_5'].rolling(window=24).mean()

print(" Added Rolling Average Features (6h, 24h).")

 Added Rolling Average Features (6h, 24h).


## 🧹 5. Final Cleanup & Export
Adding lags creates `NaN` values at the beginning of the dataset. We remove these before saving.

In [6]:
initial_len = len(df)
df = df.dropna()
final_len = len(df)

print(f"Dropped {initial_len - final_len} rows due to lag/rolling NaNs.")
print(f"Final Feature Set Shape: {df.shape}")

feature_path = ROOT_DIR / "backend/data/engineered_features.csv"
df.to_csv(feature_path, index=False)

print(f" SUCCESS: Feature Engineering complete. Data saved to {feature_path}")

Dropped 24 rows due to lag/rolling NaNs.
Final Feature Set Shape: (13152, 26)
 SUCCESS: Feature Engineering complete. Data saved to C:\Users\Dell\Desktop\AirLyst\backend\data\engineered_features.csv
